# Conception de l'architecture Hybrid SQL + FTS + RAG

## Objectif

Ce notebook est consacré à la conception de l'architecture
de recherche hybride pour les données de stages.

Il ne réalise pas encore l'implémentation du système.
L'objectif est de définir progressivement :

- le rôle du niveau ALPHA ;
- le rôle du niveau BETA ;
- l'utilisation de SQL ;
- l'utilisation de la recherche lexicale (FTS5) ;
- l'utilisation de la recherche sémantique (RAG) ;
- le flux de traitement d'une requête utilisateur.

## 1. Inspection des données pour la conception d'ALPHA

Avant de définir précisément les filtres qu'ALPHA pourra extraire
des questions utilisateur, nous inspectons les valeurs réellement
présentes dans les principales colonnes structurées du dataset nettoyé.

Cette étape est uniquement exploratoire : aucune modification des
données ne sera effectuée.

In [9]:
from pathlib import Path
import pandas as pd

# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "stages_cleaned.xlsx"

# ============================================================
# CHARGEMENT DES DONNÉES
# ============================================================

df = pd.read_excel(DATA_PATH)

print(f"Fichier chargé : {DATA_PATH.resolve()}")
print(f"Shape : {df.shape}")

Fichier chargé : C:\Users\LENOVO YOGA SLIM 6i\Desktop\rag-pfe-data\data\processed\stages_cleaned.xlsx
Shape : (2292, 24)


In [10]:
df.head()


,annee_academique,n_convention_de_stage,date_debut,date_fin,duree_totale_mois,type_de_stage,nature_du_stage,n_etudiant,nom_de_l_eleve,prenom_de_l_eleve,...,code_postal,ville,entreprise_lieu_stage,cp_lieu_stage,ville_lieu_stage,pays_lieu_stage,date_soutenance,referent_pedagogique_mail,referent_pedagogique_mail_1,duree_calculee_mois
0,NaN,SIA - 0 -,NaT,NaT,0.0,SIA,NaN,2025133,FEUILLET,Kerian,...,EX5 2FN,EXETER,NaN,NaN,NaN,NaN,NaT,sarah.bure@supmicrotech.fr,BURE Sarah,NaN
1,NaN,PFE - 0 -,NaT,NaT,0.0,PFE,NaN,2024195,PRIGENT,Capucine,...,1347,LE SENTIER,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
2,NaN,PFE - 4522 -,NaT,NaT,0.0,PFE,NaN,2024223,DEXEMPLE,Thibault,...,2017,BOUDRY,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
3,NaN,SIB - 0 -,NaT,NaT,0.0,SIB,NaN,2025212,VERET,Louis,...,32901-6975,MELBOURNE,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN
4,2020/2021,SIA - 1784 - 2020/2021,2020-07-06,2020-12-24,5.7,SIA,Stage en entreprise,2019213,FIMBEL,Nicolas,...,CH-4030,BASEL EUROAIRPORT,JET AVIATION,74100,COLLONGES,FRANCE,2021-02-03,joel.imbaud@supmicrotech.fr,IMBAUD Joël,5.617608


In [11]:
df.columns.tolist()

['annee_academique',
 'n_convention_de_stage',
 'date_debut',
 'date_fin',
 'duree_totale_mois',
 'type_de_stage',
 'nature_du_stage',
 'n_etudiant',
 'nom_de_l_eleve',
 'prenom_de_l_eleve',
 'entreprise',
 'titre_du_stage',
 'descriptif_du_stage',
 'adresse',
 'code_postal',
 'ville',
 'entreprise_lieu_stage',
 'cp_lieu_stage',
 'ville_lieu_stage',
 'pays_lieu_stage',
 'date_soutenance',
 'referent_pedagogique_mail',
 'referent_pedagogique_mail_1',
 'duree_calculee_mois']

In [12]:
# ============================================================
# INSPECTION DES VALEURS POUR LA CONCEPTION D'ALPHA
# ============================================================

columns_to_inspect = [
    "annee_academique",
    "type_de_stage",
    "nature_du_stage",
    "entreprise",
    "ville",
    "ville_lieu_stage",
    "pays_lieu_stage",
]

for col in columns_to_inspect:
    print("\n" + "=" * 70)
    print(f"COLONNE : {col}")
    print("=" * 70)

    print(f"Nombre de valeurs uniques : {df[col].nunique(dropna=True)}")
    print(f"Valeurs manquantes : {df[col].isna().sum()}")

    print("\nValeurs les plus fréquentes :")
    print(df[col].value_counts(dropna=False).head(30))


COLONNE : annee_academique
Nombre de valeurs uniques : 7
Valeurs manquantes : 4

Valeurs les plus fréquentes :
annee_academique
2023/2024    379
2022/2023    375
2021/2022    369
2020/2021    360
2025/2026    355
2024/2025    350
2026/2027    100
NaN            4
Name: count, dtype: int64

COLONNE : type_de_stage
Nombre de valeurs uniques : 8
Valeurs manquantes : 0

Valeurs les plus fréquentes :
type_de_stage
PFE           963
SIB           562
SIA           530
MASTER         73
SNO            60
SD             51
PFE_MASTER     46
PFE_CPRO        7
Name: count, dtype: int64

COLONNE : nature_du_stage
Nombre de valeurs uniques : 5
Valeurs manquantes : 453

Valeurs les plus fréquentes :
nature_du_stage
Stage en entreprise                    1525
NaN                                     453
Stage en laboratoire                    243
Stage Plateforme SUPMICROTECH-ENSMM      52
Contrat de professionnalisation          18
Formation continue                        1
Name: count, dtype: int

In [13]:
# ============================================================
# INSPECTION DES COLONNES COMPLÉMENTAIRES
# ============================================================

columns_to_inspect = [
    "date_debut",
    "date_fin",
    "duree_totale_mois",
    "date_soutenance",
    "titre_du_stage",
    "descriptif_du_stage",
]

for col in columns_to_inspect:
    print("\n" + "=" * 70)
    print(f"COLONNE : {col}")
    print("=" * 70)

    print(f"Type : {df[col].dtype}")
    print(f"Valeurs uniques : {df[col].nunique(dropna=True)}")
    print(f"Valeurs manquantes : {df[col].isna().sum()}")

    if col in ["titre_du_stage", "descriptif_du_stage"]:
        print("\nLongueur des textes :")
        print(df[col].dropna().astype(str).str.len().describe())

        print("\nExemples :")
        print(df[col].dropna().head(5).to_string(index=False))

    else:
        print("\nStatistiques :")
        print(df[col].describe())


COLONNE : date_debut
Type : datetime64[us]
Valeurs uniques : 359
Valeurs manquantes : 11

Statistiques :
count                          2281
mean     2023-08-17 12:38:11.626479
min             2020-06-29 00:00:00
25%             2022-02-07 00:00:00
50%             2023-09-01 00:00:00
75%             2025-02-17 00:00:00
max             2027-02-15 00:00:00
Name: date_debut, dtype: object

COLONNE : date_fin
Type : datetime64[us]
Valeurs uniques : 453
Valeurs manquantes : 12

Statistiques :
count                          2280
mean     2024-01-24 18:39:47.368421
min             2020-11-06 00:00:00
25%             2022-07-29 00:00:00
50%             2024-01-31 00:00:00
75%             2025-08-01 00:00:00
max             2027-07-11 00:00:00
Name: date_fin, dtype: object

COLONNE : duree_totale_mois
Type : float64
Valeurs uniques : 145
Valeurs manquantes : 0

Statistiques :
count    2292.000000
mean        5.330606
std         1.148828
min         0.000000
25%         5.000000
50%         5.

In [14]:
structured_fields = {
    "annee_academique": "structured",
    "n_convention_de_stage": "structured",
    "date_debut": "structured",
    "date_fin": "structured",
    "duree_totale_mois": "structured",
    "type_de_stage": "structured",
    "nature_du_stage": "structured",
    "n_etudiant": "structured",
    "nom_de_l_eleve": "structured",
    "prenom_de_l_eleve": "structured",
    "entreprise": "structured",
    "titre_du_stage": "text",
    "descriptif_du_stage": "text",
    "adresse": "structured",
    "code_postal": "structured",
    "ville": "structured",
    "entreprise_lieu_stage": "structured",
    "cp_lieu_stage": "structured",
    "ville_lieu_stage": "structured",
    "pays_lieu_stage": "structured",
    "date_soutenance": "structured",
    "referent_pedagogique_mail": "structured",
    "referent_pedagogique_mail_1": "structured",
    "duree_calculee_mois": "technical"
}

for field, field_type in structured_fields.items():
    print(f"{field:<35} → {field_type}")

annee_academique                    → structured
n_convention_de_stage               → structured
date_debut                          → structured
date_fin                            → structured
duree_totale_mois                   → structured
type_de_stage                       → structured
nature_du_stage                     → structured
n_etudiant                          → structured
nom_de_l_eleve                      → structured
prenom_de_l_eleve                   → structured
entreprise                          → structured
titre_du_stage                      → text
descriptif_du_stage                 → text
adresse                             → structured
code_postal                         → structured
ville                               → structured
entreprise_lieu_stage               → structured
cp_lieu_stage                       → structured
ville_lieu_stage                    → structured
pays_lieu_stage                     → structured
date_soutenance                 

In [15]:
missing_from_classification = set(df.columns) - set(structured_fields)
extra_in_classification = set(structured_fields) - set(df.columns)

print("Colonnes non classées :", missing_from_classification)
print("Champs inexistants :", extra_in_classification)

Colonnes non classées : set()
Champs inexistants : set()


In [16]:
structured_columns = [
    field
    for field, field_type in structured_fields.items()
    if field_type == "structured"
]

print("Champs structurés utilisables par ALPHA :")
for field in structured_columns:
    print("-", field)

Champs structurés utilisables par ALPHA :
- annee_academique
- n_convention_de_stage
- date_debut
- date_fin
- duree_totale_mois
- type_de_stage
- nature_du_stage
- n_etudiant
- nom_de_l_eleve
- prenom_de_l_eleve
- entreprise
- adresse
- code_postal
- ville
- entreprise_lieu_stage
- cp_lieu_stage
- ville_lieu_stage
- pays_lieu_stage
- date_soutenance
- referent_pedagogique_mail
- referent_pedagogique_mail_1


In [17]:
alpha_candidate_fields = [
    "annee_academique",
    "date_debut",
    "date_fin",
    "duree_totale_mois",
    "type_de_stage",
    "nature_du_stage",
    "n_etudiant",
    "nom_de_l_eleve",
    "prenom_de_l_eleve",
    "entreprise",
    "ville_lieu_stage",
    "pays_lieu_stage",
    "date_soutenance"
]

In [18]:
secondary_structured_fields = [
    "n_convention_de_stage",
    "adresse",
    "code_postal",
    "ville",
    "entreprise_lieu_stage",
    "cp_lieu_stage",
    "referent_pedagogique_mail",
    "referent_pedagogique_mail_1"
]

In [19]:
text_fields = [
    "titre_du_stage",
    "descriptif_du_stage"
]

In [20]:
technical_fields = [
    "duree_calculee_mois"
]

In [21]:
print("=== CHAMPS ALPHA ===")
for field in alpha_candidate_fields:
    print(f"  - {field}")

print("\n=== CHAMPS STRUCTURÉS SECONDAIRES ===")
for field in secondary_structured_fields:
    print(f"  - {field}")

print("\n=== CHAMPS TEXTUELS → BETA ===")
for field in text_fields:
    print(f"  - {field}")

print("\n=== CHAMPS TECHNIQUES ===")
for field in technical_fields:
    print(f"  - {field}")

=== CHAMPS ALPHA ===
  - annee_academique
  - date_debut
  - date_fin
  - duree_totale_mois
  - type_de_stage
  - nature_du_stage
  - n_etudiant
  - nom_de_l_eleve
  - prenom_de_l_eleve
  - entreprise
  - ville_lieu_stage
  - pays_lieu_stage
  - date_soutenance

=== CHAMPS STRUCTURÉS SECONDAIRES ===
  - n_convention_de_stage
  - adresse
  - code_postal
  - ville
  - entreprise_lieu_stage
  - cp_lieu_stage
  - referent_pedagogique_mail
  - referent_pedagogique_mail_1

=== CHAMPS TEXTUELS → BETA ===
  - titre_du_stage
  - descriptif_du_stage

=== CHAMPS TECHNIQUES ===
  - duree_calculee_mois


## 4.1 Spécification des champs structurés ALPHA

Pour chaque champ retenu, nous définissons :

- la colonne source ;
- le type de valeur attendu ;
- les formulations naturelles que l'utilisateur peut employer ;
- la normalisation nécessaire avant utilisation ;
- le type de filtre SQL applicable.

In [22]:
alpha_field_spec = {
    "annee_academique": {
        "column": "annee_academique",
        "type": "academic_year",
        "normalization": "mapping",
        "sql_operators": ["="]
    },

    "date_debut": {
        "column": "date_debut",
        "type": "date",
        "normalization": "date",
        "sql_operators": ["=", "<", "<=", ">", ">="]
    },

    "date_fin": {
        "column": "date_fin",
        "type": "date",
        "normalization": "date",
        "sql_operators": ["=", "<", "<=", ">", ">="]
    },

    "duree_totale_mois": {
        "column": "duree_totale_mois",
        "type": "numeric",
        "normalization": "numeric",
        "sql_operators": ["=", "<", "<=", ">", ">="]
    },

    "type_de_stage": {
        "column": "type_de_stage",
        "type": "category",
        "normalization": "mapping",
        "sql_operators": ["="]
    },

    "nature_du_stage": {
        "column": "nature_du_stage",
        "type": "category",
        "normalization": "mapping",
        "sql_operators": ["="]
    },

    "n_etudiant": {
        "column": "n_etudiant",
        "type": "identifier",
        "normalization": "text",
        "sql_operators": ["="]
    },

    "nom_de_l_eleve": {
        "column": "nom_de_l_eleve",
        "type": "text",
        "normalization": "text",
        "sql_operators": ["="]
    },

    "prenom_de_l_eleve": {
        "column": "prenom_de_l_eleve",
        "type": "text",
        "normalization": "text",
        "sql_operators": ["="]
    },

    "entreprise": {
        "column": "entreprise",
        "type": "text",
        "normalization": "text",
        "sql_operators": ["="]
    },

    "ville_lieu_stage": {
        "column": "ville_lieu_stage",
        "type": "text",
        "normalization": "text",
        "sql_operators": ["="]
    },

    "pays_lieu_stage": {
        "column": "pays_lieu_stage",
        "type": "text",
        "normalization": "text",
        "sql_operators": ["="]
    },

    "date_soutenance": {
        "column": "date_soutenance",
        "type": "date",
        "normalization": "date",
        "sql_operators": ["=", "<", "<=", ">", ">="]
    }
}

In [23]:
for field, spec in alpha_field_spec.items():
    print(
        f"{field:<25} | "
        f"type={spec['type']:<12} | "
        f"normalization={spec['normalization']:<8} | "
        f"operators={spec['sql_operators']}"
    )

annee_academique          | type=academic_year | normalization=mapping  | operators=['=']
date_debut                | type=date         | normalization=date     | operators=['=', '<', '<=', '>', '>=']
date_fin                  | type=date         | normalization=date     | operators=['=', '<', '<=', '>', '>=']
duree_totale_mois         | type=numeric      | normalization=numeric  | operators=['=', '<', '<=', '>', '>=']
type_de_stage             | type=category     | normalization=mapping  | operators=['=']
nature_du_stage           | type=category     | normalization=mapping  | operators=['=']
n_etudiant                | type=identifier   | normalization=text     | operators=['=']
nom_de_l_eleve            | type=text         | normalization=text     | operators=['=']
prenom_de_l_eleve         | type=text         | normalization=text     | operators=['=']
entreprise                | type=text         | normalization=text     | operators=['=']
ville_lieu_stage          | type=text      

## 4.2 Valeurs possibles des champs catégoriels

Avant de définir les règles d'interprétation d'ALPHA, nous examinons
les valeurs réellement présentes dans le dataset.

Cela permettra ensuite de construire les mappings entre les formulations
naturelles des utilisateurs et les valeurs canoniques du dataset.

In [24]:
categorical_fields = [
    "annee_academique",
    "type_de_stage",
    "nature_du_stage"
]

for field in categorical_fields:
    print("=" * 70)
    print(f"CHAMP : {field}")
    print("=" * 70)
    print(df[field].value_counts(dropna=False).to_string())
    print()

CHAMP : annee_academique
annee_academique
2023/2024    379
2022/2023    375
2021/2022    369
2020/2021    360
2025/2026    355
2024/2025    350
2026/2027    100
NaN            4

CHAMP : type_de_stage
type_de_stage
PFE           963
SIB           562
SIA           530
MASTER         73
SNO            60
SD             51
PFE_MASTER     46
PFE_CPRO        7

CHAMP : nature_du_stage
nature_du_stage
Stage en entreprise                    1525
NaN                                     453
Stage en laboratoire                    243
Stage Plateforme SUPMICROTECH-ENSMM      52
Contrat de professionnalisation          18
Formation continue                        1



## 4.3 Mapping du type de stage

Le champ `type_de_stage` contient des codes internes au système.

ALPHA doit pouvoir reconnaître les formulations naturelles utilisées
par l'utilisateur et les convertir vers les valeurs canoniques présentes
dans le dataset.

Les valeurs canoniques sont :

- `PFE`
- `SIB`
- `SIA`
- `MASTER`
- `SNO`
- `SD`
- `PFE_MASTER`
- `PFE_CPRO`

In [25]:
type_de_stage_values = sorted(
    df["type_de_stage"].dropna().unique()
)

print(type_de_stage_values)

['MASTER', 'PFE', 'PFE_CPRO', 'PFE_MASTER', 'SD', 'SIA', 'SIB', 'SNO']


## 4.4 Mapping de la nature du stage

Le champ `nature_du_stage` contient les catégories suivantes :

- `Stage en entreprise`
- `Stage en laboratoire`
- `Stage Plateforme SUPMICROTECH-ENSMM`
- `Contrat de professionnalisation`
- `Formation continue`

Les valeurs manquantes sont conservées et ne sont pas interprétées
automatiquement.

In [26]:
nature_du_stage_values = sorted(
    df["nature_du_stage"].dropna().unique()
)

print(nature_du_stage_values)

['Contrat de professionnalisation', 'Formation continue', 'Stage Plateforme SUPMICROTECH-ENSMM', 'Stage en entreprise', 'Stage en laboratoire']


### 4.4 Schéma de sortie d'ALPHA

ALPHA analyse la question en langage naturel et produit une sortie structurée.
Cette sortie contient la requête originale, les filtres structurés identifiés,
la partie textuelle destinée à BETA, ainsi que la route du traitement.

### 4.4.1 Valeurs autorisées dans la sortie d'ALPHA

La sortie d'ALPHA doit respecter un schéma contrôlé afin de pouvoir être
validée et transformée de manière déterministe en requête SQL et/ou en
requête textuelle pour BETA.

Les champs et opérateurs autorisés sont définis à partir du schéma du
dataset.

In [29]:
alpha_output_example = {
    "original_query": "Quels stages en France chez THALES concernent le machine learning ?",
    
    "route": "SQL_BETA",
    
    "structured_filters": [
        {
            "field": "pays_lieu_stage",
            "operator": "=",
            "value": "FRANCE"
        },
        {
            "field": "entreprise",
            "operator": "=",
            "value": "THALES"
        }
    ],
    
    "text_query": "machine learning"
}

alpha_output_example

{'original_query': 'Quels stages en France chez THALES concernent le machine learning ?',
 'route': 'SQL_BETA',
 'structured_filters': [{'field': 'pays_lieu_stage',
   'operator': '=',
   'value': 'FRANCE'},
  {'field': 'entreprise', 'operator': '=', 'value': 'THALES'}],
 'text_query': 'machine learning'}

In [30]:
allowed_routes = [
    "SQL",
    "SQL_BETA",
    "BETA"
]

allowed_operators = [
    "=",
    "<",
    "<=",
    ">",
    ">="
]

allowed_fields = list(alpha_field_spec.keys())

print("Routes :", allowed_routes)
print("Opérateurs :", allowed_operators)
print("Champs :", allowed_fields)

Routes : ['SQL', 'SQL_BETA', 'BETA']
Opérateurs : ['=', '<', '<=', '>', '>=']
Champs : ['annee_academique', 'date_debut', 'date_fin', 'duree_totale_mois', 'type_de_stage', 'nature_du_stage', 'n_etudiant', 'nom_de_l_eleve', 'prenom_de_l_eleve', 'entreprise', 'ville_lieu_stage', 'pays_lieu_stage', 'date_soutenance']


### 4.4.2 Types de valeurs attendus

Chaque champ structuré possède un type de valeur attendu.
Cette information permettra de valider la sortie d'ALPHA avant de construire
la requête SQL.

In [31]:
alpha_value_types = {
    "annee_academique": "academic_year",
    "date_debut": "date",
    "date_fin": "date",
    "duree_totale_mois": "numeric",
    "type_de_stage": "category",
    "nature_du_stage": "category",
    "n_etudiant": "identifier",
    "nom_de_l_eleve": "text",
    "prenom_de_l_eleve": "text",
    "entreprise": "text",
    "ville_lieu_stage": "text",
    "pays_lieu_stage": "text",
    "date_soutenance": "date"
}

for field, value_type in alpha_value_types.items():
    print(f"{field} → {value_type}")

annee_academique → academic_year
date_debut → date
date_fin → date
duree_totale_mois → numeric
type_de_stage → category
nature_du_stage → category
n_etudiant → identifier
nom_de_l_eleve → text
prenom_de_l_eleve → text
entreprise → text
ville_lieu_stage → text
pays_lieu_stage → text
date_soutenance → date


In [32]:
secondary_structured_fields = [
    "n_convention_de_stage",
    "adresse",
    "code_postal",
    "ville",
    "entreprise_lieu_stage",
    "cp_lieu_stage",
    "referent_pedagogique_mail",
    "referent_pedagogique_mail_1"
]

### 4.4.3 Règles de validation des valeurs

La valeur extraite par ALPHA doit respecter le type attendu par le champ.
La validation est effectuée après la réponse du LLM, avant toute génération
de requête SQL.

In [33]:
alpha_value_rules = {
    "academic_year": {
        "python_type": "str",
        "format": "YYYY/YYYY"
    },
    "date": {
        "python_type": "str",
        "format": "YYYY-MM-DD"
    },
    "numeric": {
        "python_type": "int_or_float",
        "format": "numeric"
    },
    "category": {
        "python_type": "str",
        "format": "canonical_dataset_value"
    },
    "identifier": {
        "python_type": "str",
        "format": "text"
    },
    "text": {
        "python_type": "str",
        "format": "text"
    }
}

alpha_value_rules

{'academic_year': {'python_type': 'str', 'format': 'YYYY/YYYY'},
 'date': {'python_type': 'str', 'format': 'YYYY-MM-DD'},
 'numeric': {'python_type': 'int_or_float', 'format': 'numeric'},
 'category': {'python_type': 'str', 'format': 'canonical_dataset_value'},
 'identifier': {'python_type': 'str', 'format': 'text'},
 'text': {'python_type': 'str', 'format': 'text'}}

### 4.4.4 Structure des filtres structurés

Chaque contrainte structurée identifiée par ALPHA est représentée par un
objet contenant trois éléments : le champ concerné, l'opérateur de comparaison
et la valeur à utiliser.

Plusieurs filtres peuvent être présents dans une même requête.

In [34]:
alpha_filter_example = {
    "field": "duree_totale_mois",
    "operator": ">",
    "value": 6
}

alpha_filter_example

{'field': 'duree_totale_mois', 'operator': '>', 'value': 6}

### 4.4.5 Schéma de sortie d'ALPHA

ALPHA extrait les contraintes structurées et l'intention textuelle de la
question. Il ne détermine pas directement la route de traitement.

La route est calculée de manière déterministe après validation de la sortie
d'ALPHA.

In [35]:
alpha_output_example = {
    "original_query": "Quels stages en France chez THALES concernent le machine learning ?",
    
    "structured_filters": [
        {
            "field": "pays_lieu_stage",
            "operator": "=",
            "value": "FRANCE"
        },
        {
            "field": "entreprise",
            "operator": "=",
            "value": "THALES"
        }
    ],
    
    "text_query": "machine learning"
}

alpha_output_example

{'original_query': 'Quels stages en France chez THALES concernent le machine learning ?',
 'structured_filters': [{'field': 'pays_lieu_stage',
   'operator': '=',
   'value': 'FRANCE'},
  {'field': 'entreprise', 'operator': '=', 'value': 'THALES'}],
 'text_query': 'machine learning'}

### 4.4.6 Définition de `text_query`

`text_query` représente la partie de la question qui doit être traitée par
BETA à travers la recherche textuelle et/ou sémantique.

Il ne s'agit pas nécessairement d'une simple liste de mots-clés. Il peut
conserver une expression ou une formulation suffisamment riche pour
préserver l'intention de recherche de l'utilisateur.

Lorsque la question ne contient aucune recherche textuelle, `text_query`
vaut `None`.

In [36]:
text_query_examples = [
    {
        "question": "Quels stages en France chez THALES concernent le machine learning ?",
        "text_query": "machine learning"
    },
    {
        "question": "Quels stages de plus de 6 mois portent sur la cybersécurité ?",
        "text_query": "cybersécurité"
    },
    {
        "question": "Je cherche des stages en laboratoire en Suède liés à l'analyse de données médicales.",
        "text_query": "analyse de données médicales"
    },
    {
        "question": "Quels stages sont disponibles en France ?",
        "text_query": None
    }
]

for example in text_query_examples:
    print("Question :", example["question"])
    print("text_query :", example["text_query"])
    print()

Question : Quels stages en France chez THALES concernent le machine learning ?
text_query : machine learning

Question : Quels stages de plus de 6 mois portent sur la cybersécurité ?
text_query : cybersécurité

Question : Je cherche des stages en laboratoire en Suède liés à l'analyse de données médicales.
text_query : analyse de données médicales

Question : Quels stages sont disponibles en France ?
text_query : None



### 4.4.7 Règle de présence de `text_query`

`text_query` est renseigné lorsque la question contient une intention de
recherche portant sur le titre ou le descriptif du stage.

Il vaut `None` lorsque la question peut être entièrement traitée à partir
des champs structurés.

La présence de `text_query` est déterminée par ALPHA, puis contrôlée lors
de la validation de sa sortie.

In [37]:
text_query_rules_examples = [
    {
        "question": "Quels stages sont disponibles en France ?",
        "text_query": None
    },
    {
        "question": "Quels stages en France concernent le machine learning ?",
        "text_query": "machine learning"
    },
    {
        "question": "Quels stages en laboratoire en Suède sont liés à la cybersécurité ?",
        "text_query": "cybersécurité"
    },
    {
        "question": "Quels stages chez THALES ont commencé après 2025 ?",
        "text_query": None
    }
]

for example in text_query_rules_examples:
    print(f"Question : {example['question']}")
    print(f"text_query : {example['text_query']}")
    print()

Question : Quels stages sont disponibles en France ?
text_query : None

Question : Quels stages en France concernent le machine learning ?
text_query : machine learning

Question : Quels stages en laboratoire en Suède sont liés à la cybersécurité ?
text_query : cybersécurité

Question : Quels stages chez THALES ont commencé après 2025 ?
text_query : None



### 4.4.8 Schéma complet de sortie d'ALPHA

La sortie d'ALPHA contient trois éléments :

- `original_query` : la question originale de l'utilisateur ;
- `structured_filters` : les contraintes pouvant être traitées par SQL ;
- `text_query` : la partie de la question destinée à la recherche textuelle
  et sémantique de BETA.

La route (`SQL`, `SQL_BETA` ou `BETA`) n'est pas produite par le LLM.
Elle sera déterminée de manière déterministe après validation.

In [38]:
alpha_output_schema_example = {
    "original_query": "...",
    
    "structured_filters": [
        {
            "field": "...",
            "operator": "...",
            "value": "..."
        }
    ],
    
    "text_query": "..."
}

alpha_output_schema_example

{'original_query': '...',
 'structured_filters': [{'field': '...', 'operator': '...', 'value': '...'}],
 'text_query': '...'}

### 4.4.9 Validation de la structure générale

Avant de vérifier le contenu des filtres, on vérifie que la sortie d'ALPHA
possède la structure attendue.

In [39]:
required_alpha_keys = [
    "original_query",
    "structured_filters",
    "text_query"
]

def validate_alpha_structure(alpha_output):
    if not isinstance(alpha_output, dict):
        return False, "La sortie d'ALPHA doit être un dictionnaire."

    missing_keys = [
        key for key in required_alpha_keys
        if key not in alpha_output
    ]

    if missing_keys:
        return False, f"Clés manquantes : {missing_keys}"

    return True, "Structure valide."


is_valid, message = validate_alpha_structure(alpha_output_schema_example)

print(message)

Structure valide.


In [40]:
invalid_alpha_output = {
    "original_query": "...",
    "structured_filters": []
}

is_valid, message = validate_alpha_structure(invalid_alpha_output)

print(message)

Clés manquantes : ['text_query']


### 4.4.10 Validation de `structured_filters`

`structured_filters` doit être une liste. Chaque élément de cette liste doit
être un dictionnaire contenant exactement les informations nécessaires à la
construction d'une condition SQL : `field`, `operator` et `value`.

In [41]:
def validate_structured_filters(alpha_output):
    filters = alpha_output["structured_filters"]

    if not isinstance(filters, list):
        return False, "`structured_filters` doit être une liste."

    for i, filter_item in enumerate(filters):
        if not isinstance(filter_item, dict):
            return False, f"Le filtre {i} doit être un dictionnaire."

        required_filter_keys = [
            "field",
            "operator",
            "value"
        ]

        missing_keys = [
            key for key in required_filter_keys
            if key not in filter_item
        ]

        if missing_keys:
            return False, (
                f"Filtre {i} : clés manquantes : {missing_keys}"
            )

    return True, "structured_filters valide."

In [42]:
valid_filters_output = {
    "original_query": "Quels stages durent plus de 6 mois ?",
    "structured_filters": [
        {
            "field": "duree_totale_mois",
            "operator": ">",
            "value": 6
        }
    ],
    "text_query": None
}

is_valid, message = validate_structured_filters(valid_filters_output)

print(message)

structured_filters valide.


In [43]:
invalid_filters_output = {
    "original_query": "Quels stages durent plus de 6 mois ?",
    "structured_filters": [
        {
            "field": "duree_totale_mois",
            "value": 6
        }
    ],
    "text_query": None
}

is_valid, message = validate_structured_filters(invalid_filters_output)

print(message)

Filtre 0 : clés manquantes : ['operator']


### 4.4.11 Validation du champ

Le champ produit par ALPHA doit appartenir à la liste des champs structurés
autorisés.

In [44]:
def validate_filter_fields(alpha_output):
    filters = alpha_output["structured_filters"]

    for i, filter_item in enumerate(filters):
        field = filter_item["field"]

        if field not in allowed_fields:
            return False, (
                f"Filtre {i} : champ non autorisé : '{field}'"
            )

    return True, "Tous les champs sont autorisés."

In [45]:
valid_field_output = {
    "original_query": "Quels stages durent plus de 6 mois ?",
    "structured_filters": [
        {
            "field": "duree_totale_mois",
            "operator": ">",
            "value": 6
        }
    ],
    "text_query": None
}

is_valid, message = validate_filter_fields(valid_field_output)

print(message)

Tous les champs sont autorisés.


In [46]:
invalid_field_output = {
    "original_query": "Quels stages durent plus de 6 mois ?",
    "structured_filters": [
        {
            "field": "salaire",
            "operator": ">",
            "value": 1000
        }
    ],
    "text_query": None
}

is_valid, message = validate_filter_fields(invalid_field_output)

print(message)

Filtre 0 : champ non autorisé : 'salaire'


### 4.5 Implémentation d'ALPHA

ALPHA utilise un LLM pour analyser la question utilisateur et produire une
sortie structurée contenant les contraintes SQL et la requête textuelle
destinée à BETA.

Le LLM ne génère pas directement de SQL et ne choisit pas la route technique.

In [47]:
import os
from dotenv import load_dotenv

load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

print("Clé chargée :", NVIDIA_API_KEY is not None)

Clé chargée : True


In [48]:
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

In [50]:
response = client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b",
    messages=[
        {
            "role": "user",
            "content": "Réponds simplement : connexion réussie."
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

Connexion réussie.


In [59]:
alpha_system_prompt = """
Tu es ALPHA, le module d'analyse des requêtes d'un système hybride
SQL + FTS5 + RAG pour rechercher des stages d'étudiants.

Ta mission est d'analyser la question de l'utilisateur et de la transformer
en une structure JSON contrôlée.

Tu dois identifier :

1. Les contraintes structurées qui peuvent être traitées par SQL.
2. La partie de la question qui nécessite une recherche textuelle
   dans le titre ou le descriptif du stage.

Tu ne dois PAS générer de SQL.
Tu ne dois PAS choisir de route.
Tu ne dois PAS répondre à la question de l'utilisateur.
Tu dois uniquement produire le JSON demandé.

--------------------------------------------------
CHAMPS STRUCTURÉS AUTORISÉS
--------------------------------------------------

annee_academique
date_debut
date_fin
duree_totale_mois
type_de_stage
nature_du_stage
n_etudiant
nom_de_l_eleve
prenom_de_l_eleve
entreprise
ville_lieu_stage
pays_lieu_stage
date_soutenance

--------------------------------------------------
OPÉRATEURS AUTORISÉS
--------------------------------------------------

=
<
<=
>
>=

--------------------------------------------------
CHAMPS TEXTUELS
--------------------------------------------------

titre_du_stage
descriptif_du_stage

La recherche textuelle peut porter sur ces deux champs.

--------------------------------------------------
VALEURS CATÉGORIELLES DU DATASET
--------------------------------------------------

Pour les champs catégoriels, utilise uniquement les valeurs
canoniques suivantes.

type_de_stage :
- MASTER
- PFE
- PFE_CPRO
- PFE_MASTER
- SD
- SIA
- SIB
- SNO

nature_du_stage :
- Stage en entreprise
- Stage en laboratoire
- Stage Plateforme SUPMICROTECH-ENSMM
- Contrat de professionnalisation
- Formation continue

Lorsqu'une formulation naturelle de l'utilisateur correspond
clairement à l'une de ces valeurs, utilise la valeur canonique
correspondante dans "structured_filters".

Exemple :

"stages en laboratoire"

doit être interprété comme :

{
  "field": "nature_du_stage",
  "operator": "=",
  "value": "Stage en laboratoire"
}

et "laboratoire" ne doit pas être placé dans "text_query"
dans ce cas.

De même :

"stages de type PFE"

doit être interprété comme :

{
  "field": "type_de_stage",
  "operator": "=",
  "value": "PFE"
}
--------------------------------------------------
FORMAT DE SORTIE OBLIGATOIRE
--------------------------------------------------

Retourne uniquement un objet JSON valide avec exactement ces trois clés :

{
  "original_query": "...",
  "structured_filters": [],
  "text_query": null
}

"original_query" :
La question originale de l'utilisateur, sans modification.

"structured_filters" :
Une liste de filtres structurés.
Chaque filtre doit avoir exactement :

{
  "field": "...",
  "operator": "...",
  "value": "..."
}

Si aucun filtre structuré n'est présent, retourne une liste vide.

"text_query" :
La partie de la question qui doit être recherchée dans
le titre ou le descriptif du stage.

Si aucune recherche textuelle n'est nécessaire, retourne null.

--------------------------------------------------
RÈGLES IMPORTANTES
--------------------------------------------------

- N'invente jamais un champ qui n'est pas dans la liste autorisée.
- N'invente jamais un opérateur.
- Ne transforme pas une notion textuelle en filtre SQL.
- Une notion comme "machine learning", "cybersécurité" ou
  "analyse de données médicales" doit normalement être placée dans
  "text_query".
- Conserve le sens de la question originale.
- Si la question contient à la fois des filtres structurés et une
  recherche textuelle, retourne les deux.
- Pour une question uniquement structurée, "text_query" doit être null.
- Pour une question uniquement textuelle, "structured_filters" doit être [].
- Retourne uniquement le JSON, sans explication, sans Markdown,
  sans ```json.
"""

In [52]:
test_query = "Quels stages en France chez THALES concernent le machine learning ?"

response = client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b",
    messages=[
        {
            "role": "system",
            "content": alpha_system_prompt
        },
        {
            "role": "user",
            "content": test_query
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

{
  "original_query": "Quels stages en France chez THALES concernent le machine learning ?",
  "structured_filters": [
    {
      "field": "pays_lieu_stage",
      "operator": "=",
      "value": "France"
    },
    {
      "field": "entreprise",
      "operator": "=",
      "value": "THALES"
    }
  ],
  "text_query": "machine learning"
}


In [62]:
def run_alpha(query):
    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b",
        messages=[
            {
                "role": "system",
                "content": alpha_system_prompt
            },
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [63]:
alpha_test_queries = [
    "Je cherche des stages en laboratoire en Suède liés à l'analyse de données médicales.",
    "Quels stages de type PFE concernent le développement web ?",
    "Quels stages de plus de 6 mois portent sur la cybersécurité ?",
    "Quels stages en France chez THALES concernent le machine learning ?",
    "Quels stages ont commencé entre janvier et juin 2025 ?"
]

for query in alpha_test_queries:
    print("=" * 80)
    print("QUESTION :", query)
    print()
    print(run_alpha(query))
    print()

QUESTION : Je cherche des stages en laboratoire en Suède liés à l'analyse de données médicales.

{
  "original_query": "Je cherche des stages en laboratoire en Suède liés à l'analyse de données médicales.",
  "structured_filters": [
    {
      "field": "nature_du_stage",
      "operator": "=",
      "value": "Stage en laboratoire"
    },
    {
      "field": "pays_lieu_stage",
      "operator": "=",
      "value": "Suède"
    }
  ],
  "text_query": "analyse de données médicales"
}

QUESTION : Quels stages de type PFE concernent le développement web ?

{
  "original_query": "Quels stages de type PFE concernent le développement web ?",
  "structured_filters": [
    {
      "field": "type_de_stage",
      "operator": "=",
      "value": "PFE"
    }
  ],
  "text_query": "développement web"
}

QUESTION : Quels stages de plus de 6 mois portent sur la cybersécurité ?

{
  "original_query": "Quels stages de plus de 6 mois portent sur la cybersécurité ?",
  "structured_filters": [
    {
      

In [64]:
import json

def parse_alpha_response(response_text):
    return json.loads(response_text)

In [65]:
alpha_result = parse_alpha_response(
    run_alpha("Quels stages de plus de 6 mois portent sur la cybersécurité ?")
)

print(alpha_result)
print(type(alpha_result))

{'original_query': 'Quels stages de plus de 6 mois portent sur la cybersécurité ?', 'structured_filters': [{'field': 'duree_totale_mois', 'operator': '>', 'value': '6'}], 'text_query': 'cybersécurité'}
<class 'dict'>


In [66]:
def normalize_filter_value(field, value):
    field_type = alpha_field_spec[field]["type"]

    if field_type == "numeric":
        return float(value) if "." in str(value) else int(value)

    if field_type == "date":
        return str(value)

    if field_type in ["academic_year", "category", "identifier", "text"]:
        return str(value)

    raise ValueError(f"Type de champ non géré : {field_type}")

In [67]:
print(normalize_filter_value("duree_totale_mois", "6"))
print(type(normalize_filter_value("duree_totale_mois", "6")))

print(normalize_filter_value("date_debut", "2025-01-01"))
print(normalize_filter_value("pays_lieu_stage", "France"))

6
<class 'int'>
2025-01-01
France


In [68]:
def normalize_alpha_filters(alpha_output):
    normalized_filters = []

    for filter_item in alpha_output["structured_filters"]:
        field = filter_item["field"]
        operator = filter_item["operator"]
        value = filter_item["value"]

        normalized_value = normalize_filter_value(field, value)

        normalized_filters.append({
            "field": field,
            "operator": operator,
            "value": normalized_value
        })

    return normalized_filters

In [69]:
normalized_filters = normalize_alpha_filters(alpha_result)

print(normalized_filters)

[{'field': 'duree_totale_mois', 'operator': '>', 'value': 6}]


In [70]:
print(type(normalized_filters[0]["value"]))

<class 'int'>


In [71]:
def validate_alpha_output(alpha_output):
    # 1. Vérifier la structure générale
    is_valid, message = validate_alpha_structure(alpha_output)
    if not is_valid:
        return False, message

    # 2. Vérifier la structure des filtres
    is_valid, message = validate_structured_filters(alpha_output)
    if not is_valid:
        return False, message

    # 3. Vérifier les champs autorisés
    is_valid, message = validate_filter_fields(alpha_output)
    if not is_valid:
        return False, message

    return True, "Sortie ALPHA valide."

In [72]:
is_valid, message = validate_alpha_output(alpha_result)

print(message)

Sortie ALPHA valide.


In [73]:
def determine_route(alpha_output):
    has_structured_filters = bool(alpha_output["structured_filters"])
    has_text_query = alpha_output["text_query"] is not None

    if has_structured_filters and not has_text_query:
        return "SQL"

    if has_structured_filters and has_text_query:
        return "SQL_BETA"

    if not has_structured_filters and has_text_query:
        return "BETA"

    return None

In [74]:
test_sql = {
    "original_query": "Quels stages sont disponibles en France ?",
    "structured_filters": [
        {
            "field": "pays_lieu_stage",
            "operator": "=",
            "value": "France"
        }
    ],
    "text_query": None
}

test_sql_beta = {
    "original_query": "Quels stages en France concernent l'IA ?",
    "structured_filters": [
        {
            "field": "pays_lieu_stage",
            "operator": "=",
            "value": "France"
        }
    ],
    "text_query": "intelligence artificielle"
}

test_beta = {
    "original_query": "Quels stages concernent l'intelligence artificielle ?",
    "structured_filters": [],
    "text_query": "intelligence artificielle"
}

print(determine_route(test_sql))
print(determine_route(test_sql_beta))
print(determine_route(test_beta))

SQL
SQL_BETA
BETA


In [75]:
print(determine_route(alpha_result))

SQL_BETA
